## Supplementary Table 7

Enviorment: env 1

In [16]:
import pandas as pd
import pickle
import os
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve, auc, precision_recall_curve
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix
from scipy.stats import sem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

import sys
sys.path.append("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/src/analysis/final_figures/final_submission/")
from utils import plot_avg_roc_from_dicts, plot_avg_pr_from_dicts

In [17]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
plt.rcParams["font.family"] = "DejaVu Sans"

dpi = 300
font_size = 12
fig_size = 5

### PEC performance-optimized data

In [18]:
pw1_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw1/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw2_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw2/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw3_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw3/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw4_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw4/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_8}_oof.pkl"
pw5_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw5/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_12,n_feats_32}_oof.pkl"
pw6_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw6/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_8}_oof.pkl"
pw7_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw7/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_1,learning_rate_0.1,subsample_0.8,scale_pos_weight_2,n_feats_32}_oof.pkl"
pw8_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw8/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_2,n_feats_8}_oof.pkl"
pw9_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw9/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_32}_oof.pkl"
pw10_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw10/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_1,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_32}_oof.pkl"

all_loo_paths = [pw1_5, pw2_5, pw3_5, pw4_5, pw5_5, pw6_5, pw7_5, pw8_5, pw9_5, pw10_5]

In [19]:
cu_all_probs = []
for i in range(10):
    full_path = all_loo_paths[i]
    with open(full_path, "rb") as f:
        data = pickle.load(f)
    cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [20]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

In [21]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")

In [22]:
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

In [23]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [24]:
# number subjects with predictions
len(cu_all_probs[~cu_all_probs[0].isna() | ~cu_all_probs[1].isna() | ~cu_all_probs[2].isna() | ~cu_all_probs[3].isna() | ~cu_all_probs[4].isna() | ~cu_all_probs[5].isna() |
    ~cu_all_probs[6].isna() | ~cu_all_probs[7].isna() | ~cu_all_probs[8].isna() | ~cu_all_probs[9].isna()])

1137

In [25]:
# number of pec cases with predictions
len(cu_all_probs[~cu_all_probs[0].isna() & ~cu_all_probs[1].isna() & ~cu_all_probs[2].isna() & ~cu_all_probs[3].isna() & ~cu_all_probs[4].isna() & ~cu_all_probs[5].isna() &
    ~cu_all_probs[6].isna() & ~cu_all_probs[7].isna() & ~cu_all_probs[8].isna() & ~cu_all_probs[9].isna()]) - 82

54

In [26]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    all_pec = pickle.load(f)

In [27]:
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]
sum(cu_all_probs["label"])

54

In [28]:
rename_dict = {}

for i in range(10):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [29]:
best_auc = 0
best_ap = 0
best_pp = 0
best_fp = 0
best_age = 0
for pp in np.arange(0, 1.05, 0.05):
    for fp in np.arange(0, 1.05, 0.05):
        for age in np.arange(30, 40, 1):
                reweight = cu_all_probs.copy()
                cols = ["Combo"]
                reweight["Combo"] = (reweight["Past PEC"] * pp + fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > age).astype(int)))
                auc = plot_avg_roc_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {pp} {fp}, {age})", add_col=cols, plot=False, pw=10)
                ap = plot_avg_pr_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {pp} {fp} {age})", add_col=cols, plot=False, pw=10)
                if auc > best_auc:
                    best_auc = auc
                    best_ap = ap
                    best_pp = pp
                    best_fp = fp
                    best_age = age
"auc", best_auc, "ap", best_ap, "pp", best_pp, "fp", best_fp, "age", best_age
# ('auc',
#  0.907910931408199,
#  'ap',
#  0.8075887489615529,
#  'pp',
#  0.35000000000000003,
#  'fp',
#  0.15000000000000002,
#  'age',
#  37)

('auc',
 0.907910931408199,
 'ap',
 0.8075887489615529,
 'pp',
 0.35000000000000003,
 'fp',
 0.15000000000000002,
 'age',
 37)

### PEC stability-optimized data

#### fmf

In [30]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

#### maternal information

In [31]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

#### retinal information

In [32]:
all_mets = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw/run_20260407_stable/meta_probs_auc_0.5/all_pw_metrics.csv")
result = all_mets.loc[all_mets.groupby('pw')['auc'].idxmax(), ['pw', 'hp', 'auc']]

In [33]:
cu_all_probs = []
for pw in range(1, 6):
    hp = result[result["pw"] == pw]["hp"].iloc[0]
    path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw{pw}/run_20260407_stable/meta_probs_auc_0.5/XGBoost/{hp}_oof.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
        cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [34]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [35]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    all_pec = pickle.load(f)
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]

rename_dict = {}

for i in range(6):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [36]:
all_probs = cu_all_probs[~cu_all_probs["pw_1"].isna() | ~cu_all_probs["pw_2"].isna() | ~cu_all_probs["pw_3"].isna() | ~cu_all_probs["pw_4"].isna() | ~cu_all_probs["pw_5"].isna()]
all_probs = all_probs[all_probs["label"] != 1]
len(all_probs) # how many controls

1134

In [37]:
best_auc = 0
best_ap = 0
best_pp = 0
best_fp = 0
best_age = 0
for pp in np.arange(0, 1.05, 0.05):
    for fp in np.arange(0, 1.05, 0.05):
        for age in np.arange(30, 40, 1):
                reweight = cu_all_probs.copy()
                cols = ["Combo"]
                reweight["Combo"] = (reweight["Past PEC"] * pp + fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > age).astype(int)))
                auc = plot_avg_roc_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {pp} {fp}, {age})", add_col=cols, plot=False, pw=5)
                ap = plot_avg_pr_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {pp} {fp} {age})", add_col=cols, plot=False, pw=5)
                if auc > best_auc:
                    best_auc = auc
                    best_ap = ap
                    best_pp = pp
                    best_fp = fp
                    best_age = age
"auc", best_auc, "ap", best_ap, "pp", best_pp, "fp", best_fp, "age", best_age
# ('auc',
#  0.7789180679196621,
#  'ap',
#  0.5056893008028068,
#  'pp',
#  0.35000000000000003,
#  'fp',
#  0.15000000000000002,
#  'age',
#  37)


('auc',
 0.7789180679196621,
 'ap',
 0.5056893008028068,
 'pp',
 0.35000000000000003,
 'fp',
 0.15000000000000002,
 'age',
 37)

### GHTN performance-optimized

In [38]:
ght_param = "{n_estimators_250,max_depth_3,learning_rate_0.01,subsample_1.0,scale_pos_weight_2,n_feats_8}"
ght1 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw1/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght2 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw2/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght3 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw3/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght4 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw4/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght5 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw5/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght6 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw6/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght7 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw7/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght8 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw8/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght9 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw9/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"
ght10 = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw10/run_20251020/meta_probs_auc_0.5/XGBoost/{ght_param}_oof.pkl"

all_loo_paths = [ght1, ght2, ght3, ght4, ght5, ght6, ght7, ght8, ght9, ght10]

In [39]:
cu_all_probs = []
for i in range(10):
    full_path = all_loo_paths[i]
    print(full_path)
    with open(full_path, "rb") as f:
        data = pickle.load(f)
    cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw1/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_3,learning_rate_0.01,subsample_1.0,scale_pos_weight_2,n_feats_8}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw2/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_3,learning_rate_0.01,subsample_1.0,scale_pos_weight_2,n_feats_8}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw3/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_3,learning_rate_0.01,subsample_1.0,scale_pos_weight_2,n_feats_8}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw4/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_3,learning_rate_0.01,subsample_1.0,scale_pos_weight_2,n_feats_8}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_ght_hc_pw5/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max

In [41]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

In [42]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")

In [43]:
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

In [44]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [45]:
len(cu_all_probs[~cu_all_probs[0].isna() | ~cu_all_probs[1].isna() | ~cu_all_probs[2].isna() | ~cu_all_probs[3].isna() | ~cu_all_probs[4].isna() | ~cu_all_probs[5].isna() |
    ~cu_all_probs[6].isna() | ~cu_all_probs[7].isna() | ~cu_all_probs[8].isna() | ~cu_all_probs[9].isna()])

1132

In [47]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/"
ght_cases = f"{data_path}/ght_cases_20250926.pkl"
with open(ght_cases, "rb") as f:
    ght_cases = pickle.load(f)
    ght_cases = [case.lower() for case in ght_cases]
len(ght_cases)

49

In [48]:
len(cu_all_probs[~cu_all_probs[0].isna() & ~cu_all_probs[1].isna() & ~cu_all_probs[2].isna() & ~cu_all_probs[3].isna() & ~cu_all_probs[4].isna() & ~cu_all_probs[5].isna() &
    ~cu_all_probs[6].isna() & ~cu_all_probs[7].isna() & ~cu_all_probs[8].isna() & ~cu_all_probs[9].isna()]) - 82

49

In [49]:
cu_all_probs["label"] = [int(id in ght_cases) for id in cu_all_probs["id"]]

In [50]:
rename_dict = {}

for i in range(10):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [51]:
best_auc = 0
best_ap = 0
best_fp = 0
best_age = 0
for fp in np.arange(0, 1.05, 0.05):
    for age in np.arange(30, 40, 1):
            reweight = cu_all_probs.copy()
            cols = ["Combo"]
            reweight["Combo"] = (fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > age).astype(int)))
            auc = plot_avg_roc_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp}, {age})", add_col=cols, plot=False, pw=5)
            ap = plot_avg_pr_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp} {age})", add_col=cols, plot=False, pw=5)
            if auc > best_auc:
                best_auc = auc
                best_ap = ap
                best_fp = fp
                best_age = age
"auc", best_auc, "ap", best_ap, "fp", best_fp, "age", best_age
# ('auc', 0.9437387637454548, 'ap', 0.8756829104469329, 'fp', 0.05, 'age', 34)

('auc', 0.9437387637454548, 'ap', 0.8756829104469329, 'fp', 0.05, 'age', 34)

### GHTN stability-optimized

#### maternal information

In [52]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

#### retinal information

In [53]:
all_mets = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_ghtn_strat_hc_pw/run_20260525_stable/meta_probs_auc_0.5/all_pw_metrics.csv")
result = all_mets.loc[all_mets.groupby('pw')['auc'].idxmax(), ['pw', 'hp', 'auc']]

In [54]:
cu_all_probs = []
for pw in range(1, 6):
    hp = result[result["pw"] == pw]["hp"].iloc[0]
    path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_ghtn_strat_hc_pw{pw}/run_20260525_stable/meta_probs_auc_0.5/XGBoost/{hp}_oof.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
        cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [55]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")

In [56]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/ght_cases_20250926.pkl", "rb") as f:
    ghtn = pickle.load(f)

ghtn = [id.lower() for id in ghtn]
cu_all_probs["label"] = [int(id in ghtn) for id in cu_all_probs["id"]]

rename_dict = {}

for i in range(6):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [57]:
best_auc = 0
best_ap = 0
best_fp = 0
best_age = 0
for fp in np.arange(0, 1.05, 0.05):
    for age in np.arange(30, 40, 1):
            reweight = cu_all_probs.copy()
            cols = ["Combo"]
            reweight["Combo"] = (fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > age).astype(int)))
            auc = plot_avg_roc_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp}, {age})", add_col=cols, plot=False, pw=5)
            ap = plot_avg_pr_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp} {age})", add_col=cols, plot=False, pw=5)
            if auc > best_auc:
                best_auc = auc
                best_ap = ap
                best_fp = fp
                best_age = age
"auc", best_auc, "ap", best_ap, "fp", best_fp, "age", best_age
# ('auc', 0.7955191563038646, 'ap', 0.4983638055700962, 'fp', 0.0, 'age', 30)

('auc', 0.7955191563038646, 'ap', 0.4983638055700962, 'fp', 0.0, 'age', 30)

### CHTN performance-optimized

In [58]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia"

cht1 = f"{data_path}/final_cht_hc_pw1/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_100,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_2,n_feats_8}}_oof.pkl"
cht2 = f"{data_path}/final_cht_hc_pw2/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_2,n_feats_8}}_oof.pkl"
cht3 = f"{data_path}/final_cht_hc_pw3/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_32}}_oof.pkl"
cht4 = f"{data_path}/final_cht_hc_pw4/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_8,n_feats_32}}_oof.pkl"
cht5 = f"{data_path}/final_cht_hc_pw5/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_3,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_8}}_oof.pkl"
cht6 = f"{data_path}/final_cht_hc_pw6/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_2,n_feats_8}}_oof.pkl"
cht7 = f"{data_path}/final_cht_hc_pw7/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_32}}_oof.pkl"
cht8 = f"{data_path}/final_cht_hc_pw8/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_8,n_feats_8}}_oof.pkl"
cht9 = f"{data_path}/final_cht_hc_pw9/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_12,n_feats_32}}_oof.pkl"
cht10 = f"{data_path}/final_cht_hc_pw10/run_20251024/meta_probs_auc_0.5/XGBoost/{{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_32}}_oof.pkl"

all_loo_paths = [cht1, cht2, cht3, cht4, cht5, cht6, cht7, cht8, cht9, cht10]

In [59]:
cu_all_probs = []
for i in range(10):
    full_path = all_loo_paths[i]
    print(full_path)
    with open(full_path, "rb") as f:
        data = pickle.load(f)
    cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_cht_hc_pw1/run_20251024/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_2,n_feats_8}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_cht_hc_pw2/run_20251024/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_2,n_feats_8}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_cht_hc_pw3/run_20251024/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_32}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_cht_hc_pw4/run_20251024/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_8,n_feats_32}_oof.pkl
/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_cht_hc_pw5/run_20251024/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_d

In [60]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

In [61]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")

In [62]:
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

In [63]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [64]:
len(cu_all_probs[~cu_all_probs[0].isna() | ~cu_all_probs[1].isna() | ~cu_all_probs[2].isna() | ~cu_all_probs[3].isna() | ~cu_all_probs[4].isna() | ~cu_all_probs[5].isna() |
    ~cu_all_probs[6].isna() | ~cu_all_probs[7].isna() | ~cu_all_probs[8].isna() | ~cu_all_probs[9].isna()])

1106

In [66]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/"
cht_cases = f"{data_path}/cht_cases_20250926.pkl"
with open(cht_cases, "rb") as f:
    cht_cases = pickle.load(f)
    cht_cases = [case.lower() for case in cht_cases]
len(cht_cases)

61

In [67]:
len(cu_all_probs[~cu_all_probs[0].isna() & ~cu_all_probs[1].isna() & ~cu_all_probs[2].isna() & ~cu_all_probs[3].isna() & ~cu_all_probs[4].isna() & ~cu_all_probs[5].isna() &
    ~cu_all_probs[6].isna() & ~cu_all_probs[7].isna() & ~cu_all_probs[8].isna() & ~cu_all_probs[9].isna()]) - 82

61

In [68]:
cu_all_probs["label"] = [int(id in cht_cases) for id in cu_all_probs["id"]]

In [69]:
rename_dict = {}

for i in range(10):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [ ]:
best_auc = 0
best_ap = 0
best_fp = 0
best_age = 0
for fp in np.arange(0, 1.05, 0.05):
    for age in np.arange(30, 40, 1):
            reweight = cu_all_probs.copy()
            cols = ["Combo"]
            reweight["Combo"] = (fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > age).astype(int)))
            auc = plot_avg_roc_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp}, {age})", add_col=cols, plot=False, pw=5)
            ap = plot_avg_pr_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp} {age})", add_col=cols, plot=False, pw=5)
            if auc > best_auc:
                best_auc = auc
                best_ap = ap
                best_fp = fp
                best_age = age
"auc", best_auc, "ap", best_ap, "fp", best_fp, "age", best_age
# ('auc',
#  0.9013482948073095,
#  'ap',
#  0.7545085100215398,
#  'fp',
#  0.15000000000000002,
#  'age',
#  36)

('auc',
 0.9013482948073095,
 'ap',
 0.7545085100215398,
 'fp',
 0.15000000000000002,
 'age',
 36)

### CHTN stability-optimized

#### maternal information

In [71]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

### retinal information

In [72]:
all_mets = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_chtn_strat_hc_pw/run_20260525_stable/meta_probs_auc_0.5/all_pw_metrics.csv")
result = all_mets.loc[all_mets.groupby('pw')['auc'].idxmax(), ['pw', 'hp', 'auc']]

In [73]:
cu_all_probs = []
for pw in range(1, 6):
    hp = result[result["pw"] == pw]["hp"].iloc[0]
    path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_chtn_strat_hc_pw{pw}/run_20260525_stable/meta_probs_auc_0.5/XGBoost/{hp}_oof.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
        cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [74]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")

In [75]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/cht_cases_20250926.pkl", "rb") as f:
    chtn = pickle.load(f)

chtn = [id.lower() for id in chtn]
cu_all_probs["label"] = [int(id in chtn) for id in cu_all_probs["id"]]

rename_dict = {}

for i in range(6):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [76]:
len(cu_all_probs[~cu_all_probs["pw_1"].isna() | ~cu_all_probs["pw_2"].isna() | ~cu_all_probs["pw_3"].isna() | ~cu_all_probs["pw_4"].isna() | ~cu_all_probs["pw_5"].isna()])

1143

In [77]:
len(cu_all_probs[~cu_all_probs["pw_1"].isna() & ~cu_all_probs["pw_2"].isna() & ~cu_all_probs["pw_3"].isna() & ~cu_all_probs["pw_4"].isna() & ~cu_all_probs["pw_5"].isna()])

61

In [78]:
sum(cu_all_probs["label"])

61

In [79]:
best_auc = 0
best_ap = 0
best_fp = 0
best_age = 0
for fp in np.arange(0, 1.05, 0.05):
    for age in np.arange(30, 40, 1):
        reweight = cu_all_probs.copy()
        cols = ["Combo"]
        reweight["Combo"] = (fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > age).astype(int)))
        auc = plot_avg_roc_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp}, {age})", add_col=cols, plot=False, pw=5)
        ap = plot_avg_pr_from_dicts(reweight, title=f"AVG(Retinal,  {cols} * {fp} {age})", add_col=cols, plot=False, pw=5)
        if auc > best_auc:
            best_auc = auc
            best_ap = ap
            best_fp = fp
            best_age = age
"auc", best_auc, "ap", best_ap, "fp", best_fp, "age", best_age
# ('auc', 0.7781995702833875, 'ap', 0.5310370345737042, 'fp', 0.25, 'age', 30)

('auc', 0.7781995702833875, 'ap', 0.5310370345737042, 'fp', 0.25, 'age', 30)